# Confluence Cloud REST API — Regulatory Management Data Access POC

## Objective
Validate the Confluence data-access layer required for the Regulatory Management AI solution.

| ID | Test | What it proves |
|---|---|---|
| T1 | Authentication & Connectivity | Python can reach Confluence and authenticate |
| T2 | Space Discovery | We can discover spaces visible to the identity |
| T3 | CQL Content Search | We can find relevant pages |
| T4 | Page Retrieval & Content Extraction | We can retrieve and extract actual page content |
| T5 | Pagination | We can retrieve multiple result batches without duplicate IDs |
| T6 | Metadata & Provenance | We can retain source metadata required for AI/RAG traceability |
| T7 | Access Control | REST access follows the authenticated user's Confluence permissions |

The IT Forge examples you supplied are treated as API-reference examples. This notebook uses Python `requests` because our target is an external data-consumption path.

**Not in this first test:** audit creation/export, page archival, blueprint publishing, draft publishing, and other write/admin operations.

# 0. Authentication — do this before running the tests

For a script/POC, Atlassian documents **HTTP Basic Authentication with your Atlassian account email + API token** for direct Confluence REST API calls.

## Step 1 — Create an API token

Open:

https://id.atlassian.com/manage-profile/security/api-tokens

Then:

1. Sign in with the Atlassian account you will use for the test.
2. Select **Create API token**.
3. Name it `RM-Confluence-REST-POC`.
4. Set an appropriate expiry.
5. Create it.
6. **Copy the token immediately and store it securely.** Atlassian does not let you recover the token value later.

Atlassian currently allows token expiry from **1 to 365 days**.

Official instructions:
https://support.atlassian.com/atlassian-account/docs/manage-api-tokens-for-your-atlassian-account/

## Step 2 — Confirm Confluence access

Using the same Atlassian account, confirm in the browser that you can open the Confluence spaces/pages you want to test.

The REST API follows normal Confluence permissions.

## Step 3 — Get your Confluence URL

Example:

`https://your-company.atlassian.net`

The notebook automatically uses:

`https://your-company.atlassian.net/wiki/rest/api`

## Step 4 — Check with IT

Before using the token, confirm that:

- API-token access is permitted by the HSBC/Atlassian security policy.
- Your account is allowed to call the Confluence REST API.
- Your Jupyter/Workbench environment allows outbound HTTPS.
- A corporate proxy/firewall does not need additional configuration.
- You have permission to test the intended Confluence spaces.

**Do not paste the token into a notebook code cell.** This notebook collects it with `getpass`.

### If HSBC requires scoped tokens

Atlassian's current documentation distinguishes scoped API tokens from the direct-site Basic Auth pattern. Scoped-token calls use the Atlassian API gateway and require the appropriate scopes/cloud ID.

If HSBC mandates scoped tokens only, **ask IT for the approved scoped-token/OAuth/service-account pattern rather than bypassing the policy**.

# 1. Dependencies

In [ ]:
%pip install -q requests pandas beautifulsoup4 urllib3

In [ ]:
import json
import getpass
from typing import Any, Dict, List, Optional
from urllib.parse import urljoin, urlparse

import requests
import pandas as pd
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

print("Environment ready.")

# 2. Secure configuration

Enter the three values below. The token is hidden using `getpass` and is never written to the notebook by this code.

In [ ]:
CONFLUENCE_BASE_URL = input(
    "Confluence base URL (e.g. https://company.atlassian.net): "
).strip().rstrip("/")

CONFLUENCE_EMAIL = input(
    "Atlassian account email: "
).strip()

CONFLUENCE_API_TOKEN = getpass.getpass(
    "Atlassian API token (hidden): "
)

API_PREFIX = "/wiki/rest/api"
CONFLUENCE_API_BASE = f"{CONFLUENCE_BASE_URL}{API_PREFIX}"

parsed = urlparse(CONFLUENCE_BASE_URL)

if parsed.scheme not in {"http", "https"} or not parsed.netloc:
    raise ValueError(
        "Invalid URL. Expected https://your-company.atlassian.net"
    )

if not CONFLUENCE_EMAIL:
    raise ValueError("Email cannot be empty.")

if not CONFLUENCE_API_TOKEN:
    raise ValueError("API token cannot be empty.")

print("\nConfiguration loaded.")
print("Confluence:", CONFLUENCE_BASE_URL)
print("API base  :", CONFLUENCE_API_BASE)
print("Account   :", CONFLUENCE_EMAIL)
print("Token     : [hidden]")

# 3. Reusable REST client

The IT code uses Forge's `requestConfluence()`. This client provides the equivalent external Python capability with Basic authentication, persistent sessions, connection pooling, timeout, retries, and centralised error handling.

This POC is intentionally **read-only**.

In [ ]:
class ConfluenceAPIError(RuntimeError):
    pass


class ConfluenceClient:
    """Read-only Confluence Cloud REST client for the POC."""

    def __init__(
        self,
        base_url: str,
        email: str,
        api_token: str,
        timeout: int = 30,
    ):
        self.base_url = base_url.rstrip("/")
        self.api_base = f"{self.base_url}/wiki/rest/api"
        self.timeout = timeout

        self.session = requests.Session()
        self.session.auth = (email, api_token)

        self.session.headers.update({
            "Accept": "application/json",
            "User-Agent": "RM-Confluence-REST-POC/1.0",
        })

        retry = Retry(
            total=3,
            connect=3,
            read=3,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=frozenset(["GET"]),
            respect_retry_after_header=True,
        )

        adapter = HTTPAdapter(
            max_retries=retry,
            pool_connections=10,
            pool_maxsize=10,
        )

        self.session.mount("https://", adapter)
        self.session.mount("http://", adapter)

    def _build_url(self, endpoint: str) -> str:
        if endpoint.startswith(("http://", "https://")):
            return endpoint
        return urljoin(self.api_base + "/", endpoint.lstrip("/"))

    def get(
        self,
        endpoint: str,
        params: Optional[Dict[str, Any]] = None,
    ) -> requests.Response:
        return self.session.get(
            self._build_url(endpoint),
            params=params,
            timeout=self.timeout,
        )

    def get_json(
        self,
        endpoint: str,
        params: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:

        response = self.get(endpoint, params)

        if not response.ok:
            raise ConfluenceAPIError(
                f"HTTP {response.status_code}\n"
                f"URL: {response.url}\n"
                f"Response: {response.text[:2000]}"
            )

        try:
            return response.json()
        except ValueError as exc:
            raise ConfluenceAPIError(
                f"Expected JSON but received non-JSON response. "
                f"HTTP {response.status_code}; URL={response.url}"
            ) from exc

    def close(self):
        self.session.close()

In [ ]:
client = ConfluenceClient(
    base_url=CONFLUENCE_BASE_URL,
    email=CONFLUENCE_EMAIL,
    api_token=CONFLUENCE_API_TOKEN,
)

print("REST client initialised.")

# 4. T1 — Authentication & Connectivity

**Endpoint:** `GET /wiki/rest/api/space?limit=1`

- `200` → connectivity + authentication accepted
- `401` → email/token/authentication-policy problem
- `403` → identity recognised but access forbidden
- timeout/proxy error → network/environment issue

In [ ]:
def test_authentication(client: ConfluenceClient) -> Dict[str, Any]:
    try:
        response = client.get("/space", params={"limit": 1})

        result = {
            "test_id": "T1",
            "test": "Authentication & Connectivity",
            "status_code": response.status_code,
            "passed": response.ok,
            "message": "",
        }

        if response.ok:
            result["message"] = (
                "REST API reachable and authentication accepted."
            )
        elif response.status_code == 401:
            result["message"] = (
                "401 Unauthorized — check email, token, expiry and policy."
            )
        elif response.status_code == 403:
            result["message"] = (
                "403 Forbidden — authentication may work but access is denied."
            )
        else:
            result["message"] = response.text[:500]

        return result

    except requests.RequestException as exc:
        return {
            "test_id": "T1",
            "test": "Authentication & Connectivity",
            "status_code": None,
            "passed": False,
            "message": f"Network error: {exc}",
        }


t1_result = test_authentication(client)
print(json.dumps(t1_result, indent=2))

# 5. T2 — Space Discovery

Retrieve spaces visible to the authenticated identity.

In [ ]:
def get_spaces(
    client: ConfluenceClient,
    limit: int = 100,
) -> List[Dict[str, Any]]:

    data = client.get_json(
        "/space",
        params={"limit": limit}
    )
    return data.get("results", [])


spaces = get_spaces(client)

space_df = pd.DataFrame([
    {
        "space_id": s.get("id"),
        "key": s.get("key"),
        "name": s.get("name"),
        "type": s.get("type"),
        "status": s.get("status"),
    }
    for s in spaces
])

print(f"Spaces returned: {len(space_df)}")
display(space_df)

# 6. T3 — CQL Content Search

**Endpoint:** `GET /wiki/rest/api/content/search`

The IT screenshots show `cql` as the required query parameter.

### Example 1
`type=page`

### Example 2
`type=page AND text~"regulation"`

Use a real regulatory term from your target Confluence content.

In [ ]:
def search_content(
    client: ConfluenceClient,
    cql: str,
    limit: int = 25,
    expand: Optional[str] = None,
) -> Dict[str, Any]:

    params = {
        "cql": cql,
        "limit": limit,
    }

    if expand:
        params["expand"] = expand

    return client.get_json(
        "/content/search",
        params=params
    )


t3_basic = search_content(
    client,
    cql="type=page",
    limit=10,
)

basic_results = t3_basic.get("results", [])

print(f"Basic page search returned: {len(basic_results)}")

for item in basic_results:
    print(
        item.get("id"),
        "|",
        item.get("title"),
        "|",
        item.get("type")
    )

In [ ]:
REGULATORY_SEARCH_TERM = input(
    "Enter a real regulatory search term: "
).strip()

if not REGULATORY_SEARCH_TERM:
    raise ValueError("Search term cannot be empty.")

cql = f'type=page AND text~"{REGULATORY_SEARCH_TERM}"'

t3_regulatory = search_content(
    client,
    cql=cql,
    limit=25,
)

search_results = t3_regulatory.get("results", [])

search_df = pd.DataFrame([
    {
        "page_id": item.get("id"),
        "title": item.get("title"),
        "type": item.get("type"),
        "status": item.get("status"),
        "space": (
            item.get("space", {}).get("name")
            if item.get("space")
            else None
        ),
        "webui": item.get("_links", {}).get("webui"),
    }
    for item in search_results
])

print(f"Regulatory search returned: {len(search_df)}")
display(search_df)

# 7. T4 — Page Retrieval & Content Extraction

Select the first page returned by T3 and retrieve its full page representation.

Requested expansions:

- `space`
- `version`
- `ancestors`
- `metadata.labels`
- `body.storage`

The body is converted from Confluence storage markup into plain text suitable for downstream processing.

In [ ]:
def get_page(
    client: ConfluenceClient,
    page_id: str,
    expand: Optional[str] = None,
) -> Dict[str, Any]:

    params = {}
    if expand:
        params["expand"] = expand

    return client.get_json(
        f"/content/{page_id}",
        params=params
    )


if search_df.empty:
    raise RuntimeError(
        "No search results. Re-run T3 with a known/broader term."
    )

selected_page_id = str(search_df.iloc[0]["page_id"])

page = get_page(
    client,
    selected_page_id,
    expand=(
        "space,version,ancestors,"
        "metadata.labels,body.storage"
    ),
)

print("Page ID :", page.get("id"))
print("Title   :", page.get("title"))
print("Type    :", page.get("type"))
print("Status  :", page.get("status"))
print("Space   :", (page.get("space") or {}).get("name"))
print("Version :", (page.get("version") or {}).get("number"))

In [ ]:
def html_to_text(html: str) -> str:
    if not html:
        return ""

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup.find_all(["br", "p", "div", "li", "tr"]):
        tag.append("\n")

    text = soup.get_text(separator=" ", strip=True)

    return "\n".join(
        " ".join(line.split())
        for line in text.splitlines()
        if line.strip()
    )


storage_html = (
    page
    .get("body", {})
    .get("storage", {})
    .get("value", "")
)

page_text = html_to_text(storage_html)

print("Extracted characters:", len(page_text))
print("\n--- CONTENT PREVIEW ---\n")
print(page_text[:5000])

# 8. T5 — Cursor Pagination

The dedicated Confluence search endpoint supports pagination through `next` links/cursors.

The test requests 25 records at a time, follows `next`, stops after five batches, and checks duplicate page IDs.

In [ ]:
def search_all_pages(
    client: ConfluenceClient,
    cql: str,
    page_size: int = 25,
    max_pages: int = 5,
) -> List[Dict[str, Any]]:

    results = []

    response = client.get_json(
        "/search",
        params={
            "cql": cql,
            "limit": page_size,
        }
    )

    batch_number = 1

    while True:
        batch = response.get("results", [])

        print(f"Batch {batch_number}: {len(batch)} records")
        results.extend(batch)

        next_url = response.get("_links", {}).get("next")

        if not next_url:
            print("No next cursor. Pagination complete.")
            break

        if batch_number >= max_pages:
            print(f"Safety limit reached: max_pages={max_pages}")
            break

        response = client.get_json(next_url)
        batch_number += 1

    return results


paginated_results = search_all_pages(
    client,
    cql="type=page",
    page_size=25,
    max_pages=5,
)

page_ids = [
    str(item["id"])
    for item in paginated_results
    if item.get("id") is not None
]

unique_ids = set(page_ids)
duplicate_count = len(page_ids) - len(unique_ids)

t5_result = {
    "records_retrieved": len(page_ids),
    "unique_records": len(unique_ids),
    "duplicate_ids": duplicate_count,
    "passed": duplicate_count == 0,
}

print(json.dumps(t5_result, indent=2))

# 9. T6 — Metadata & Provenance

Normalize one page into a record suitable for downstream ingestion.

Retain:

`page_id, title, type, status, space, version, last_modified, labels, ancestors, webui, text`

In [ ]:
def extract_page_record(page: Dict[str, Any]) -> Dict[str, Any]:

    space = page.get("space") or {}
    version = page.get("version") or {}

    label_results = (
        page
        .get("metadata", {})
        .get("labels", {})
        .get("results", [])
    )

    labels = [
        x.get("name")
        for x in label_results
        if x.get("name")
    ]

    ancestors = [
        {
            "id": x.get("id"),
            "title": x.get("title"),
        }
        for x in page.get("ancestors", [])
    ]

    storage_html = (
        page
        .get("body", {})
        .get("storage", {})
        .get("value", "")
    )

    text = html_to_text(storage_html)

    return {
        "page_id": page.get("id"),
        "title": page.get("title"),
        "type": page.get("type"),
        "status": page.get("status"),
        "space_id": space.get("id"),
        "space_key": space.get("key"),
        "space_name": space.get("name"),
        "version": version.get("number"),
        "last_modified": version.get("when"),
        "labels": labels,
        "ancestors": ancestors,
        "webui": page.get("_links", {}).get("webui"),
        "text": text,
        "character_count": len(text),
    }


page_record = extract_page_record(page)

metadata_view = {
    k: v
    for k, v in page_record.items()
    if k != "text"
}

print(json.dumps(metadata_view, indent=2, default=str))

# 10. T7 — Access-Control Behaviour

Use one page that you definitely can access. Optionally test a page that IT confirms you cannot access.

Do **not** guess a restricted page.

In [ ]:
def test_page_access(
    client: ConfluenceClient,
    page_id: str,
) -> Dict[str, Any]:

    response = client.get(f"/content/{page_id}")

    return {
        "page_id": page_id,
        "status_code": response.status_code,
        "accessible": response.status_code == 200,
        "classification": (
            "accessible"
            if response.status_code == 200
            else "not_accessible"
        ),
    }


accessible_test = test_page_access(
    client,
    selected_page_id
)

print(json.dumps(accessible_test, indent=2))

In [ ]:
RUN_RESTRICTED_ACCESS_TEST = input(
    "Run restricted-page test? Enter Y or N: "
).strip().upper()

restricted_test = None

if RUN_RESTRICTED_ACCESS_TEST == "Y":

    restricted_page_id = input(
        "Restricted page ID confirmed by IT: "
    ).strip()

    if not restricted_page_id:
        raise ValueError("Restricted page ID cannot be empty.")

    restricted_test = test_page_access(
        client,
        restricted_page_id
    )

    print(json.dumps(restricted_test, indent=2))
else:
    print("Restricted-page test skipped.")

# 11. Consolidated Test Report

This table is generated from actual API responses. No result is hard-coded.

In [ ]:
def status(condition: bool) -> str:
    return "PASS" if condition else "FAIL"


test_report = pd.DataFrame([
    {
        "Test ID": "T1",
        "Test": "Authentication & Connectivity",
        "Status": status(t1_result["passed"]),
        "Evidence": t1_result["message"],
    },
    {
        "Test ID": "T2",
        "Test": "Space Discovery",
        "Status": status(len(spaces) >= 0),
        "Evidence": f"{len(spaces)} spaces returned",
    },
    {
        "Test ID": "T3",
        "Test": "CQL Content Search",
        "Status": status(isinstance(search_results, list)),
        "Evidence": f"{len(search_results)} results returned",
    },
    {
        "Test ID": "T4",
        "Test": "Page Retrieval & Content Extraction",
        "Status": status(bool(page.get("id")) and bool(page_text)),
        "Evidence": f"{len(page_text)} characters extracted",
    },
    {
        "Test ID": "T5",
        "Test": "Pagination",
        "Status": status(t5_result["passed"]),
        "Evidence": (
            f"{t5_result['records_retrieved']} records; "
            f"{t5_result['duplicate_ids']} duplicate IDs"
        ),
    },
    {
        "Test ID": "T6",
        "Test": "Metadata & Provenance",
        "Status": status(
            bool(page_record["page_id"])
            and bool(page_record["title"])
            and bool(page_record["space_name"])
        ),
        "Evidence": "Normalized page metadata successfully",
    },
    {
        "Test ID": "T7",
        "Test": "Access Control",
        "Status": status(accessible_test["accessible"]),
        "Evidence": (
            f"Known-accessible page returned "
            f"HTTP {accessible_test['status_code']}"
        ),
    },
])

display(test_report)

# 12. Optional evidence export

Exports:

- `confluence_poc_test_report.csv`
- `confluence_search_results.csv`
- `confluence_selected_page_record.json`

**The API token is never exported.**

The page-record file can contain Confluence content, so handle it according to HSBC data-classification requirements.

In [ ]:
EXPORT_RESULTS = input(
    "Export evidence files? Enter Y or N: "
).strip().upper()

if EXPORT_RESULTS == "Y":

    test_report.to_csv(
        "confluence_poc_test_report.csv",
        index=False
    )

    search_df.to_csv(
        "confluence_search_results.csv",
        index=False
    )

    with open(
        "confluence_selected_page_record.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            page_record,
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    print("Created:")
    print("  confluence_poc_test_report.csv")
    print("  confluence_search_results.csv")
    print("  confluence_selected_page_record.json")

else:
    print("Export skipped.")

# 13. POC Success Criteria

The technical data-access POC passes when:

- **T1 PASS** — REST authentication/connectivity works.
- **T2 PASS** — expected spaces are visible.
- **T3 PASS** — relevant pages can be discovered with CQL.
- **T4 PASS** — actual page body can be retrieved and converted to text.
- **T5 PASS** — pagination works without duplicate IDs.
- **T6 PASS** — provenance metadata is retained.
- **T7 PASS** — access behaviour follows Confluence permissions.

### Data-access flow

```text
Confluence
    ↓
Authenticated REST access
    ↓
Space discovery
    ↓
CQL discovery
    ↓
Page retrieval
    ↓
Content extraction
    ↓
Metadata / provenance
    ↓
Paginated ingestion capability
```

The next phase, after this succeeds, is the AI ingestion layer: normalization → chunking → embeddings → RAG / regulatory entity mapping.

# 14. Troubleshooting

### 401 Unauthorized
Check the Atlassian email, token, expiry/revocation and organisation API-token policy.

### 403 Forbidden
The identity may be authenticated but lack required Confluence access.

### 404 Not Found
Check page ID, site URL, and whether the authenticated identity can see the page.

### Timeout / Proxy error
Ask IT whether the Jupyter/Workbench environment requires an HTTPS proxy, firewall exception, or outbound allow-listing.

### Search returns zero results
Start with:

`type=page`

Then:

`type=page AND text~"known-term"`

### Direct Basic Auth is blocked
Do not bypass the policy. Ask IT for the approved scoped API token + gateway, OAuth 2.0, service-account credential, or other enterprise-approved integration method.

# 15. Official Atlassian references

- Confluence Cloud REST API:
  https://developer.atlassian.com/cloud/confluence/rest/v1/intro/

- Basic authentication for REST APIs:
  https://developer.atlassian.com/cloud/confluence/basic-auth-for-rest-apis/

- Using the REST API:
  https://developer.atlassian.com/cloud/confluence/using-the-rest-api/

- API token management:
  https://support.atlassian.com/atlassian-account/docs/manage-api-tokens-for-your-atlassian-account/

In [ ]:
try:
    client.close()
    print("HTTP session closed.")
except Exception:
    pass